In [1]:
import pandas as pd
import numpy as np

import os
os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')


In [2]:
cps_raw = pd.read_csv("dataset/raw_dataset/csv_pus/psam_pusa.csv")

# #combine a and b raw dataset
cps_raw1 = pd.read_csv("dataset/raw_dataset/csv_pus/psam_pusb.csv")

cps_raw = pd.concat([cps_raw, cps_raw1], axis=0, ignore_index=True)

#need MV var in housing data and combine

house1 = pd.read_csv("dataset/raw_dataset/csv_hus/psam_husa.csv")
house2 = pd.read_csv("dataset/raw_dataset/csv_hus/psam_husb.csv")

house_merged = pd.concat([house1, house2], axis=0, ignore_index=True)
house_merged_mv = house_merged.loc[:, ["SERIALNO", "MV"]]

cps_raw = cps_raw.merge(house_merged_mv, on="SERIALNO", how="left")

In [3]:
cps_raw.shape

(3422888, 287)

In [4]:
# SELECT variables
vars = ["PWGTP","MV", "RACBLK", "ESR", "SEX", "ADJINC", "POBP", "MIGSP", "SCHL", "AGEP", "SPORDER", "PERNP", "WKHP", "WKWN", "PUMA", "STATE"]

cps_raw = cps_raw.loc[:, vars].copy()

In [5]:
cps_raw.head()

,PWGTP,MV,RACBLK,ESR,SEX,ADJINC,POBP,MIGSP,SCHL,AGEP,SPORDER,PERNP,WKHP,WKWN,PUMA,STATE
0,41,NaN,1,6.0,1,1015250,1,NaN,10.0,59,1,18500.0,30.0,52.0,2500,1
1,52,NaN,0,6.0,1,1015250,1,NaN,17.0,43,1,0.0,NaN,NaN,2600,1
2,31,NaN,1,6.0,2,1015250,1,1.0,16.0,75,1,0.0,NaN,NaN,1401,1
3,4,NaN,0,6.0,2,1015250,233,NaN,19.0,22,1,0.0,NaN,NaN,2802,1
4,19,NaN,0,6.0,1,1015250,1,1.0,16.0,51,1,0.0,NaN,NaN,800,1


In [6]:
rename = {
    "WGTP": "housing_weight",
    "PWGTP": "pums_weight",
    "MV": "mv",
    "RACBLK": "black",
    "ESR": "emp_status",
    "SEX": "sex",
    "ADJINC": "inc_adj",
    "POBP": "birth_place",
    "MIGSP": "mig_rec",
    "SCHL": "educ",
    "AGEP": "age",
    "WKHP": "hours_per_week",
    "WKWN": "weeks_per_year",
    "SPORDER": "person_num",
    "PERNP": "total_earnings"

}

cps_raw.columns = cps_raw.columns.str.strip()
cps_raw = cps_raw.rename(columns=rename)

In [7]:
(cps_raw["mv"].astype(float) > 6).mean() * 100

np.float64(11.750603583874202)

In [8]:
#when they were 14?

#create numeric value for MV variable

def organize_mv(df):

    df = df.copy()
    mv_map = {
        0: np.nan,
        1: 0.5,
        2: 1.5,
        3: 3,
        4: 7.5,
        5: 15,
        6: 25,
        7: 30
    }

    df["mv_age"] = df["mv"].map(mv_map)

    #1 if they did not move since 14 years old
    df["same_home_as_14ys"] = ((df["age"].astype(int) - df["mv_age"]) <= 14).astype(int)
    #2 13 years old
    df["same_home_as_13ys"] = ((df["age"].astype(int) - df["mv_age"]) <= 13).astype(int)
    #3 12 years old
    df["same_home_as_12ys"] = ((df["age"].astype(int) - df["mv_age"]) <= 12).astype(int)

    return df.copy()

# implement
cps_raw = organize_mv(cps_raw)

In [9]:
cps_raw["same_home_as_14ys"].mean()

np.float64(0.2372034959951947)

In [10]:
#filter earnings data
"""
earnings > 0
hours worked > 0
week worked > 0
employed == 1

"""

def wage_filter(df):
    return df[
        (df["total_earnings"] > 0) &
        (df["hours_per_week"] > 0) &
        (df["weeks_per_year"] > 0)
    ].copy()

cps_raw = wage_filter(cps_raw)

In [11]:
def cps_vars(df):

    df = df.copy()
    # female dummy
    df["female"] = (df["sex"] == 2).astype(int)

    # years of schooling
    educ_map = {
        0: np.nan,
        1: 0,
        2: 0,
        3: 0,
        4: 1,
        5: 2,
        6: 3,
        7: 4,
        8: 5,
        9: 6,
        10: 7,
        11: 8,
        12: 9,
        13: 10,
        14: 11,
        15: 11,
        16: 12,
        17: 12,
        18: 12,
        19: 13,
        20: 14,
        21: 16,
        22: 18,
        23: 18,
        24: 22
    }
    df["educ"] = df["educ"].map(educ_map)

    # log wage
    df["wage"] = (
    df["inc_adj"] * df["total_earnings"]
    ) / (df["weeks_per_year"] * df["hours_per_week"])

    df["log_wage"] = np.log(df["wage"])

    # experience
    df["exp"] = np.maximum(df["age"] - df["educ"] - 6, 0)

    #exp-square
    df["exp2"] = df["exp"]**2

    # employment status
    df["employed"] = df["emp_status"].isin([1,2]).astype(int)

    # mover dummy
    df["mover"] = (df["birth_place"] != 5).astype(int)

    return df.copy()



cps_raw = cps_vars(cps_raw)


In [12]:
# filter unemployment data
def emp_filter(df):
    return df[df["employed"] == 1].copy()

cps_raw = emp_filter(cps_raw)

In [13]:
# create age bins (categorical variable)
def age_bins(df):
    df = df.copy()

    df["age_bins"] = pd.cut(
        df["age"],
        bins=range(0, 101, 10),   # 0,10,20,...100
        right=False,              # 左閉右開区間 [ )
        labels=[f"{i}-{i+9}" for i in range(10, 101, 10)]
        )
    return df.copy()

cps_raw = age_bins(cps_raw)

In [14]:
#make state and puma code into str
cps_raw["STATE"] = cps_raw["STATE"].astype(str).str.zfill(2)
cps_raw["PUMA"] = cps_raw["PUMA"].astype(str).str.zfill(5)

# create state + puma key (PUMA key can be the same between states)
cps_raw["STATE_PUMA"] = (cps_raw["STATE"] + "_" + cps_raw["PUMA"])

In [15]:
# check cell
cps_raw.shape

(1593027, 29)

In [16]:
# export with 3 different age filter

cps_raw = cps_raw[["person_num","log_wage", "educ", "exp", "exp2", "female", "black", "mv", "age",
                   "birth_place", "employed", "PUMA", "STATE", "STATE_PUMA", "same_home_as_14ys", "same_home_as_13ys",
                   "same_home_as_12ys", "mover", "age_bins", "pums_weight"]]

cps_14 = cps_raw[cps_raw["same_home_as_14ys"] == 1]
cps_13 = cps_raw[cps_raw["same_home_as_13ys"] == 1]
cps_12 = cps_raw[cps_raw["same_home_as_12ys"] == 1]

cps_14.to_parquet("dataset/cleaned_dataset/cleaned_cps14.parquet")
cps_13.to_parquet("dataset/cleaned_dataset/cleaned_cps13.parquet")
cps_12.to_parquet("dataset/cleaned_dataset/cleaned_cps12.parquet")

In [17]:
cps_raw.columns

Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place', 'employed', 'PUMA', 'STATE', 'STATE_PUMA',
       'same_home_as_14ys', 'same_home_as_13ys', 'same_home_as_12ys', 'mover',
       'age_bins', 'pums_weight'],
      dtype='object')

In [18]:
cps_raw["PUMA"].unique()

array(['01202', '00402', '02802', ..., '02105', '02806', '02807'],
      shape=(1150,), dtype=object)

In [19]:
cps_raw["STATE_PUMA"].unique()

array(['01_01202', '01_00402', '01_02802', ..., '56_00500', '56_00200',
       '56_00300'], shape=(2462,), dtype=object)